In [ ]:
from pathlib import Path
from functools import partial

import numpy as np
import matplotlib.pyplot as plt
from nptdms import TdmsFile
from scipy.optimize import minimize, differential_evolution
from scipy.interpolate import CubicSpline
from scipy import stats

In [ ]:

# ── Channel Correction ───────────────────────────────────────────────────────
def T_Icor_Matrix():
    ret = np.zeros([4, 4])
    ret[0][0] = 0.152
    ret[0][1] = 0
    ret[1][0] = 0
    ret[1][1] = 0.148
    alpha = 0.455
    beta = 0.009
    t = 0.757
    r = 0.741
    ret[2][0] = -alpha * beta * r
    ret[2][1] = alpha * beta * r
    ret[2][2] = alpha * alpha * r
    ret[2][3] = beta * beta * r
    ret[3][0] = -alpha * beta * t
    ret[3][1] = alpha * beta * t
    ret[3][2] = beta * beta * t
    ret[3][3] = alpha * alpha * t
    return np.linalg.inv(ret) / 8

def true_best_coeff_func_mat(params, ac, mat):
    a90, a45, a135 = params
    ap = np.copy(ac)
    ap[1, :] *= a90
    ap[2, :] *= a45
    ap[3, :] *= a135
    bc = np.dot(mat, ap)
    c0 = bc[0, :]
    c90 = bc[1, :]
    c45 = bc[2, :]
    c135 = bc[3, :]
    return np.sum((c0 + c90 - c45 - c135) ** 2)

def find_best_coeff_using_mat(c0, c90, c45, c135, mat):
    ac = np.asarray(np.vstack((c0, c90, c45, c135)))
    return minimize(partial(true_best_coeff_func_mat, ac=ac, mat=mat), [1, 1, 1])


# ── Electronic offset (APD) ───────────────────────────────────────────────────
def compute_offset(c0, c90, c45, c135):
    """o = [o_90, o_45, o_135, o_0] — electronic offset estimate.
    Input channels should be gain-corrected (a_i · c_i).
    o_i = max(0, -min(c_i)) — shifts each channel so its minimum is >= 0."""
    return [max(0.0, -float(np.min(ch))) for ch in [c90, c45, c135, c0]]


# ── Anisotropy Center Correction (radial method) ─────────────────────────────
def radial_asymmetry_score(center, points, angles=None, k=3, band_width=0.05):
    if angles is None:
        angles = np.linspace(0, 2 * np.pi, 16, endpoint=False)
    center = np.asarray(center)
    scores = []
    for theta in angles:
        direction = np.array([np.cos(theta), np.sin(theta)])
        normal = np.array([-np.sin(theta), np.cos(theta)])
        rel_points = points - center
        projections_along = rel_points @ direction
        projections_orth = np.abs(rel_points @ normal)
        mask = projections_orth < band_width
        in_band = projections_along[mask]
        pos = np.sort(in_band[in_band > 0])
        neg = np.sort(in_band[in_band < 0])
        if len(pos) < k or len(neg) < k:
            continue
        d_pos = np.mean(pos[:k])
        d_neg = np.mean(neg[-k:])
        scores.append(d_pos - d_neg)
    if len(scores) == 0:
        return 1000
    return -np.mean(scores)

def find_radial_center(points):
    bounds = [(-0.5, 0.5), (-0.5, 0.5)]
    result = differential_evolution(
        lambda c: radial_asymmetry_score(c, points, k=3, band_width=0.05),
        bounds,
    )
    return result.x

def estimate_center(x, y):
    points = np.column_stack((x, y))
    return find_radial_center(points)


# ── Phi linearity correction ─────────────────────────────────────────────────
def correct_on_speed_step(m, speeds, N=500, fftfilter=1, mfilter=7, show=1):
    nn, _ = np.histogram(np.array(m) % (2 * np.pi), N)
    bins = np.linspace(0, 2 * np.pi, N)
    binned_std_displacement = stats.binned_statistic(m, speeds, "std", bins=bins)
    sy = binned_std_displacement.statistic
    if show:
        plt.figure()
        plt.plot(np.array(m[1:]) % (2 * np.pi), speeds, ".", markersize=1, label="raw points")
        plt.plot(np.linspace(0, 2 * np.pi, N), sy, label="Average signal")
    if 0 in nn:
        print("Warning: I did not correct non linearities because of undersampling")
        return lambda x: x
    if fftfilter:
        rft = np.fft.rfft(sy)
        rft[mfilter:] = 0
        sy = np.fft.irfft(rft)
        if show:
            plt.plot(np.linspace(0, 2 * np.pi, N), sy, label="Filtered signal")
            plt.legend()
            plt.xlabel("Phir (rad)")
            plt.ylabel("Value")
    fcor = np.cumsum(1 / sy)
    fcor = np.insert(fcor, 0, 0)
    fcor = fcor * 2 * np.pi / fcor[-1]
    f = CubicSpline(np.linspace(0, 2 * np.pi, N - 1), fcor)
    if show:
        plt.figure()
        plt.plot(np.linspace(0, 2 * np.pi, N - 1), f(np.linspace(0, 2 * np.pi, N - 1)), ".-")
        plt.xlabel("phi_ori")
        plt.ylabel("phi_old")
    return f

def correct_on_diff(phir, phiu, show=0, fcor=None, expanding=False):
    if fcor is None:
        if expanding:
            step = 1_000_000
            n = len(phir)
            fcor = None
            for end in range(step, n + step, step):
                end = min(end, n)
                candidate = correct_on_speed_step(phir[1:end], np.diff(phiu[:end]), show=show)
                if isinstance(candidate, CubicSpline):
                    print(f"fcor fitted on first {end:,} points (expanding window, use_expanding_window = True)")
                    fcor = candidate
                    break
                else:
                    print(f"Non-linearity not applied because of undersampling, use_expanding_window = True, re-trying non-linearity with a larger window (window size: {end:,})")
                if end >= n:
                    print("fcor: could not fit on any expanding window — phi unchanged")
                    fcor = lambda x: x
                    break
        else:
            candidate = correct_on_speed_step(phir[1:1_000_000], np.diff(phiu[:1_000_000]), show=show)
            if isinstance(candidate, CubicSpline):
                print("fcor fitted on first 1,000,000 points (use_expanding_window = False)")
                fcor = candidate
            else:
                print("Non-linearity not applied because of undersampling, use_expanding_window = False, phi unchanged")
                fcor = lambda x: x
    phirc = fcor(phir)
    phiuc = np.unwrap(phirc)
    return phirc, phiuc, fcor


# ── Anisotropy ───────────────────────────────────────────────────────────────
def anisotropy_from_channels(c0, c90, c45, c135, center=(0.0, 0.0)):
    denom_x = c0 + c90
    denom_y = c45 + c135
    ax = np.where(denom_x > 0, (c0 - c90)   / denom_x, 0.0) - center[0]
    ay = np.where(denom_y > 0, (c45 - c135) / denom_y, 0.0) - center[1]
    return ax, ay


In [ ]:

# ── File list — edit this ────────────────────────────────────────────────────
# centering_mode   : "bbox" | "radial" | "none" | "manual"
# enable_ch_cor    : True  → apply channel correction
#                    False → use raw channels as-is
# enable_t_matrix  : True  → also apply T_Icor_Matrix (requires enable_ch_cor=True)
#                    False → apply coeff scaling only (a[i] factors, no T matrix)
#
# Tuple format: (path, centering_mode, enable_ch_cor, enable_t_matrix)

tdms_file_list = [
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2022_06_14_5_strept0.tdms"),   "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2022_06_14_5_strept3.tdms"),   "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2free.tdms"),   "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2free3.tdms"),   "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2free6.tdms"),   "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2free7.tdms"),   "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\free4.tdms"),   "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\free444444452.tdms"),  "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\free4444450.tdms"),  "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\free444449.tdms"),  "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed5.tdms"),  "none", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed6.tdms"),  "none", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed7.tdms"),  "none", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed8.tdms"),  "none", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed9.tdms"),  "none", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed10.tdms"),  "none", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\Extended Figure 4 data\051225\12.tdms"),  "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\Extended Figure 4 data\051225\1114.tdms"),  "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\Extended Figure 4 data\051225\11115.tdms"),  "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\Extended Figure 4 data\091225\12131214-2147483648.tdms"),  "bbox", True, True),
    (Path(r"C:\Users\denny\workspace\pyqtrod\data\Extended Figure 4 data\091225\09122591226.tdms"),  "bbox", True, True),
]

# Manual centers — only used when centering_mode == "manual"
manual_centers = {
    # Path(r"C:\path\to\some_file.tdms"): (cx, cy),
}

# ── Settings ──────────────────────────────────────────────────────────────────
dec_plot       = 1000   # decimation for plots
dec_correction = 1000   # decimation for channel-correction fit

# Per-channel offset: o_i = max(0, -min(a_i * c_i))
# Computed automatically per file after gain (a) is determined — shifts the
# most-negative sample of each gain-corrected channel to exactly zero.

fit_start_s = 10.0   # time window for channel-correction fit
fit_end_s   = 20.0   # set to None to use full file

use_expanding_window           = True
apply_phi_linearity_correction = False

# ── Check paths ───────────────────────────────────────────────────────────────
print(f"{'File':<50} {'center mode':>12}  {'ch_cor':>6}  {'T_mat':>5}  {'exists':>6}")
print("-" * 88)
for p, cm, en_ch, en_t in tdms_file_list:
    exists = "YES" if p.exists() else "NO !"
    print(f"  {p.name:<48} {cm:>12}  {str(en_ch):>6}  {str(en_t):>5}  {exists:>6}")
print(f"\nTotal: {len(tdms_file_list)} file(s) defined.")


In [ ]:

# ── Load raw channel intensities from all TDMS files ────────────────────────
# Stores raw_data_all[path] = {c0, c90, c45, c135, time_all_s, freq, ...}
# Offset (o) is computed later, after gain (a) is determined.

orientations = ["90", "45", "135", "0"]
matcorb      = T_Icor_Matrix()

def get_pol_ind(listpol, orientations=orientations):
    out = []
    for pol in listpol:
        for j, ori in enumerate(orientations):
            if ori == pol:
                out.append(j)
                break
    return out

inds = get_pol_ind(["0", "90", "45", "135"])
Pmat = np.zeros([4, 4])
for i in range(4):
    Pmat[i, inds[i]] = 1
matcor = np.dot(np.linalg.inv(Pmat), np.dot(matcorb, Pmat))

raw_data_all = {}

for tdms_path, center_mode, enable_ch_cor, enable_t_matrix in tdms_file_list:
    print(f"\nLoading: {tdms_path.name}")
    if not tdms_path.exists():
        print(f"  FILE NOT FOUND — skipping")
        continue
    try:
        tdms_obj  = TdmsFile.open(str(tdms_path))
    except Exception as e:
        print(f"  ERROR opening file: {e} — skipping")
        continue

    group    = tdms_obj.groups()[0]
    channels = group.channels()
    datasize = len(channels[0])

    try:
        time_inc = float(channels[0].properties["wf_increment"])
        time_off = float(channels[0].properties["wf_start_offset"])
    except KeyError:
        time_inc = 4e-6
        time_off = 0.0
        print("  Warning: no time metadata, using time_inc=4e-6")

    freq       = 1.0 / time_inc
    time_all_s = time_off + time_inc * np.arange(datasize)

    idx      = get_pol_ind(["0", "90", "45", "135"])
    c0_raw   = np.asarray(channels[idx[0]][:], dtype=float)
    c90_raw  = np.asarray(channels[idx[1]][:], dtype=float)
    c45_raw  = np.asarray(channels[idx[2]][:], dtype=float)
    c135_raw = np.asarray(channels[idx[3]][:], dtype=float)

    raw_data_all[tdms_path] = dict(
        c0=c0_raw, c90=c90_raw, c45=c45_raw, c135=c135_raw,
        time_all_s=time_all_s,
        freq=freq, time_inc=time_inc, time_off=time_off,
        datasize=datasize,
        center_mode=center_mode,
        enable_ch_cor=enable_ch_cor,
        enable_t_matrix=enable_t_matrix,
    )
    print(f"  {datasize:,} points  |  {freq:.1f} Hz  |  duration: {time_all_s[-1]:.1f} s")

print(f"\nDone — loaded {len(raw_data_all)} / {len(tdms_file_list)} file(s).")


In [ ]:

# ── Apply channel correction and recentering ─────────────────────────────────
# enable_ch_cor  = False → raw channels passed through unchanged
# enable_ch_cor  = True, enable_t_matrix = False → coeff scaling only  (a[i] factors, no T matrix)
# enable_ch_cor  = True, enable_t_matrix = True  → full correction      (coeff scaling + T matrix)
#
# Order of operations:
#   1. Gain (a):        a[i] * c[i]             — per-channel scaling
#   2. Offset (o):      o[i] + a[i] * c[i]      — electronic offset from gain-corrected channels
#   3. T matrix:        matcor @ stack           — optical cross-talk correction
#   4. Background (b):  c_out[i] + b[i]         — background offset on corrected channels
#
# o, a, and b all use physical stack order: [90, 45, 135, 0]
# b_background is set to 0 for now — method to calculate this TBD.
#
# Stores corrected_data_all[path] = {c0..c135 corrected, anis_x, anis_y, center, ...}

b_background = [0.0, 0.0, 0.0, 0.0]   # [b_90, b_45, b_135, b_0]

corrected_data_all = {}

for tdms_path, raw in raw_data_all.items():
    print(f"\n{'='*60}")
    print(f"Processing: {tdms_path.name}")

    center_mode     = raw['center_mode']
    enable_ch_cor   = raw['enable_ch_cor']
    enable_t_matrix = raw['enable_t_matrix']
    datasize        = raw['datasize']
    time_inc        = raw['time_inc']
    time_off        = raw['time_off']

    c0_raw   = raw['c0']
    c90_raw  = raw['c90']
    c45_raw  = raw['c45']
    c135_raw = raw['c135']

    # ── Channel correction ────────────────────────────────────────────────────
    if not enable_ch_cor:
        c0_cor, c90_cor, c45_cor, c135_cor = c0_raw, c90_raw, c45_raw, c135_raw
        print("  Channel correction: DISABLED — raw channels used")

    else:
        # ── Channel-correction coefficients ──────────────────────────────────
        chcor_file = Path(str(tdms_path)[:-5] + "_chcor.npy")
        if chcor_file.exists():
            a      = np.load(chcor_file).astype(float).tolist()
            ch_src = "loaded"
        else:
            fit_s = (int((fit_start_s - time_off) / time_inc)
                     if fit_start_s is not None else 0)
            fit_e = (int((fit_end_s   - time_off) / time_inc)
                     if fit_end_s   is not None else datasize)
            fit_s = max(0, min(fit_s, datasize))
            fit_e = max(fit_s, min(fit_e, datasize))

            fit = find_best_coeff_using_mat(
                c0_raw  [fit_s:fit_e:dec_correction],
                c90_raw [fit_s:fit_e:dec_correction],
                c45_raw [fit_s:fit_e:dec_correction],
                c135_raw[fit_s:fit_e:dec_correction],
                matcorb)
            l90, l45, l135 = fit.x
            a = [1.0, 1.0, 1.0, 1.0]
            a[get_pol_ind(["90"])[0]]  = float(l90)
            a[get_pol_ind(["45"])[0]]  = float(l45)
            a[get_pol_ind(["135"])[0]] = float(l135)
            np.save(chcor_file, np.array(a, dtype=float))
            ch_src = "computed"

        if any(v < 0 for v in a):
            print(f"  WARNING: negative channel coefficient — {a}")

        # Offset from gain-corrected channels: o_i = max(0, -min(a_i * c_i))
        o = compute_offset(a[3]*c0_raw, a[0]*c90_raw, a[1]*c45_raw, a[2]*c135_raw)

        t_status = "ENABLED" if enable_t_matrix else "DISABLED"
        print(f"  Channel correction ({ch_src}): a={[f'{v:.4f}' for v in a]}  |  T matrix: {t_status}")
        print(f"  Offset o={[f'{v:.4g}' for v in o]}  |  Background b={[f'{v:.4g}' for v in b_background]}")

        # Steps 1–2: gain + offset → T matrix  [physical order: c90, c45, c135, c0]
        raw_stack = np.vstack((c90_raw, c45_raw, c135_raw, c0_raw)).astype(float)
        for i in range(4):
            raw_stack[i] = o[i] + a[i] * raw_stack[i]

        if enable_t_matrix:
            cor_stack = np.dot(matcor, raw_stack)
        else:
            cor_stack = raw_stack

        # Step 3: background  [same physical order: b_90, b_45, b_135, b_0]
        for i in range(4):
            cor_stack[i] += b_background[i]

        idx2 = get_pol_ind(["0", "90", "45", "135"])
        c0_cor, c90_cor, c45_cor, c135_cor = (cor_stack[idx2[k]] for k in range(4))

    # ── Centering ─────────────────────────────────────────────────────────────
    ax_sub, ay_sub = anisotropy_from_channels(
        c0_cor[::dec_correction], c90_cor[::dec_correction],
        c45_cor[::dec_correction], c135_cor[::dec_correction])

    if center_mode == "bbox":
        cx = (float(np.nanmin(ax_sub)) + float(np.nanmax(ax_sub))) / 2
        cy = (float(np.nanmin(ay_sub)) + float(np.nanmax(ay_sub))) / 2
        anisotropy_center = [cx, cy]

    elif center_mode == "radial":
        center_file = Path(str(tdms_path)[:-5] + "_anisotropy_center.npy")
        if center_file.exists():
            anisotropy_center = np.load(center_file).tolist()
        else:
            anisotropy_center = estimate_center(ax_sub, ay_sub).tolist()
            np.save(center_file, np.array(anisotropy_center))
        cx, cy = anisotropy_center

    elif center_mode == "manual":
        anisotropy_center = list(manual_centers.get(tdms_path, [0.0, 0.0]))
        cx, cy = anisotropy_center

    else:  # "none"
        cx, cy = 0.0, 0.0
        anisotropy_center = [0.0, 0.0]

    print(f"  Center ({center_mode}): cx={anisotropy_center[0]:.5f}, cy={anisotropy_center[1]:.5f}")

    # ── Full-resolution anisotropy ────────────────────────────────────────────
    anis_x, anis_y = anisotropy_from_channels(
        c0_cor, c90_cor, c45_cor, c135_cor, center=anisotropy_center)

    corrected_data_all[tdms_path] = dict(
        c0=c0_cor, c90=c90_cor, c45=c45_cor, c135=c135_cor,
        anis_x=anis_x, anis_y=anis_y,
        anisotropy_center=anisotropy_center,
        time_all_s=raw['time_all_s'],
        freq=raw['freq'],
    )

print(f"\nDone — corrected {len(corrected_data_all)} file(s).")


In [ ]:

# ── Anisotropy scatter plots — quality check ─────────────────────────────────

n_files = len(corrected_data_all)
ncols   = min(3, n_files)
nrows   = max(1, int(np.ceil(n_files / ncols))) if n_files else 1

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(4.6 * ncols, 4.2 * nrows),
                          dpi=100, constrained_layout=True,
                          squeeze=False)
axes_flat = axes.flatten()

for ax_i, (tdms_path, data) in enumerate(corrected_data_all.items()):
    ax_plot = axes_flat[ax_i]
    ax_d    = data['anis_x'][::dec_plot]
    ay_d    = data['anis_y'][::dec_plot]
    cx, cy  = data['anisotropy_center']

    ax_plot.scatter(ax_d, ay_d, s=1, alpha=0.3, c="tab:blue", rasterized=True)
    ax_plot.axhline(0, color="k", lw=0.5, zorder=3)
    ax_plot.axvline(0, color="k", lw=0.5, zorder=3)
    ax_plot.set_title(tdms_path.name, fontsize=8)
    ax_plot.set_xlabel("(I0−I90)/(I0+I90) − cx", fontsize=8)
    ax_plot.set_ylabel("(I45−I135)/(I45+I135) − cy", fontsize=8)
    ax_plot.set_xlim(-1, 1); ax_plot.set_ylim(-1, 1)
    ax_plot.set_aspect("equal"); ax_plot.grid(alpha=0.25)
    ax_plot.text(0.04, 0.96,
                 f"cx={cx:.4f}\ncy={cy:.4f}",
                 transform=ax_plot.transAxes, fontsize=7,
                 va="top", ha="left",
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))

for j in range(n_files, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Per-pair anisotropy  —  (I0−I90)/(I0+I90)  vs  (I45−I135)/(I45+I135)  [centered]",
             fontsize=9)
plt.show()


In [ ]:

# ── Compute r, smooth, and extract saturation statistics ─────────────────────
# Stores r_stats_all[path] = {r_max, r_995, r_peak, r_full, r_smooth}

SMOOTH_N = 100   # rolling-mean window (points)

r_stats_all = {}

for tdms_path, data in corrected_data_all.items():
    anis_x     = data['anis_x']
    anis_y     = data['anis_y']
    time_all_s = data['time_all_s']

    r_full   = np.sqrt(anis_x**2 + anis_y**2)
    r_smooth = np.convolve(r_full, np.ones(SMOOTH_N) / SMOOTH_N, mode='same')

    trim   = SMOOTH_N // 2
    r_trim = r_smooth[trim:-trim] if len(r_smooth) > 2 * trim else r_smooth

    r_max  = float(np.nanmax(r_trim))
    r_995  = float(np.nanpercentile(r_trim, 99.5))

    hist_counts, hist_bins = np.histogram(r_trim, bins=300, range=(0.0, 1.0))
    hist_bcs = 0.5 * (hist_bins[:-1] + hist_bins[1:])
    r_peak   = float(hist_bcs[np.argmax(hist_counts)])

    r_stats_all[tdms_path] = dict(
        r_max=r_max, r_995=r_995, r_peak=r_peak,
        r_full=r_full, r_smooth=r_smooth,
    )

    print(f"{tdms_path.name}:")
    print(f"  r_max={r_max:.5f}   r_99.5%={r_995:.5f}   r_peak={r_peak:.5f}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axs = plt.subplots(1, 2, figsize=(11, 3), dpi=100, constrained_layout=True)

    t_dec  = time_all_s[::dec_plot]
    rf_dec = r_full[::dec_plot]
    rs_dec = r_smooth[::dec_plot]

    axs[0].plot(t_dec, rf_dec, lw=0.4, alpha=0.4, color="tab:blue",   label="raw r")
    axs[0].plot(t_dec, rs_dec, lw=0.9, color="tab:orange",             label=f"smooth {SMOOTH_N} pt")
    axs[0].axhline(r_max,  color="tab:red",    lw=1.0, ls="--", label=f"r_max={r_max:.4f}")
    axs[0].axhline(r_995,  color="tab:purple", lw=1.0, ls=":",  label=f"r_99.5%={r_995:.4f}")
    axs[0].set_xlabel("Time (s)"); axs[0].set_ylabel("r")
    axs[0].set_title(f"{tdms_path.name} — r trace")
    axs[0].legend(fontsize=7, frameon=False); axs[0].grid(alpha=0.25)

    axs[1].hist(r_trim, bins=300, range=(0.0, 1.0), color="tab:blue", alpha=0.7)
    axs[1].axvline(r_max,  color="tab:red",    lw=1.5, ls="--", label=f"r_max  = {r_max:.4f}")
    axs[1].axvline(r_995,  color="tab:purple", lw=1.5, ls=":",  label=f"r_99.5%= {r_995:.4f}")
    axs[1].axvline(r_peak, color="tab:orange", lw=1.5, ls="-",  label=f"r_peak = {r_peak:.4f}")
    axs[1].set_xlabel("r (smoothed)"); axs[1].set_ylabel("Counts")
    axs[1].set_title("r histogram")
    axs[1].legend(fontsize=7, frameon=False); axs[1].grid(alpha=0.25)

    plt.show()

print("\nSummary:")
print(f"  {'File':<45} {'r_max':>8} {'r_99.5%':>9} {'r_peak':>8}")
print("  " + "-" * 72)
for tdms_path, st in r_stats_all.items():
    print(f"  {tdms_path.name:<45} {st['r_max']:>8.5f} {st['r_995']:>9.5f} {st['r_peak']:>8.5f}")


In [ ]:

# ── High-resolution detail plots for cutoff selection ────────────────────────
# Two separate figures per file: r trace (tail-zoomed) + tail histogram only.

FOCUS_NAMES = {"2free6.tdms", "free4444450.tdms"}
DEC_HR      = 10    # light decimation for the time trace
N_BINS_HR   = 1000  # histogram bins
R_ZOOM_LO   = 0.7   # lower bound for both plots

for tdms_path, st in r_stats_all.items():
    if tdms_path.name not in FOCUS_NAMES:
        continue

    data       = corrected_data_all[tdms_path]
    time_all_s = data['time_all_s']
    r_full     = st['r_full']
    r_smooth   = st['r_smooth']
    trim       = SMOOTH_N // 2
    r_trim     = r_smooth[trim:-trim] if len(r_smooth) > 2 * trim else r_smooth

    r_max  = st['r_max']
    r_995  = st['r_995']
    r_peak = st['r_peak']

    t_hr  = time_all_s[::DEC_HR]
    rf_hr = r_full[::DEC_HR]
    rs_hr = r_smooth[::DEC_HR]

    # ── Figure 1: r trace zoomed to tail ─────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(16, 4), dpi=130, constrained_layout=True)
    ax1.plot(t_hr, rf_hr, lw=0.3, alpha=0.35, color="tab:blue",  label="raw r")
    ax1.plot(t_hr, rs_hr, lw=0.8, color="tab:orange",             label=f"smooth {SMOOTH_N} pt")
    ax1.axhline(r_max,  color="tab:red",    lw=1.2, ls="--", label=f"r_max   = {r_max:.4f}")
    ax1.axhline(r_995,  color="tab:purple", lw=1.2, ls=":",  label=f"r_99.5% = {r_995:.4f}")
    ax1.axhline(r_peak, color="tab:green",  lw=1.2, ls="-.", label=f"r_peak  = {r_peak:.4f}")
    ax1.set_ylim(R_ZOOM_LO, 1.02)
    ax1.set_xlabel("Time (s)", fontsize=11)
    ax1.set_ylabel("r (smoothed)", fontsize=11)
    ax1.set_title(f"{tdms_path.name}  —  r trace (dec={DEC_HR},  y zoomed [{R_ZOOM_LO}–1])", fontsize=11)
    ax1.legend(fontsize=9, frameon=True)
    ax1.grid(alpha=0.3)
    plt.show()

    # ── Figure 2: tail histogram only ────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(10, 5), dpi=130, constrained_layout=True)
    ax2.hist(r_trim, bins=N_BINS_HR, range=(R_ZOOM_LO, 1.0),
             color="tab:blue", alpha=0.7, linewidth=0)
    ax2.axvline(r_max,  color="tab:red",    lw=1.5, ls="--", label=f"r_max   = {r_max:.4f}")
    ax2.axvline(r_995,  color="tab:purple", lw=1.5, ls=":",  label=f"r_99.5% = {r_995:.4f}")
    ax2.axvline(r_peak, color="tab:green",  lw=1.5, ls="-.", label=f"r_peak  = {r_peak:.4f}")
    ax2.set_xlim(R_ZOOM_LO, 1.0)
    ax2.set_xlabel("r (smoothed)", fontsize=11)
    ax2.set_ylabel("Counts", fontsize=11)
    ax2.set_title(f"{tdms_path.name}  —  r histogram tail [{R_ZOOM_LO}–1]  ({N_BINS_HR} bins)", fontsize=11)
    ax2.legend(fontsize=9, frameon=True)
    ax2.grid(alpha=0.3)
    plt.show()


In [ ]:

# ── Calibration: back-calculate α (NA) from measured r_sat ───────────────────
#
# Fourkas model for a dipole at polar angle θ (from optical axis) and azimuthal
# angle φ, collected by a high-NA objective with half-angle α:
#
#   I_ψ = A(α) + B(α)·sin²θ + C(α)·sin²θ·cos²(ψ − φ)
#
# where A, B, C are functions of α only (see fourkas_ABC below).
#
# The pairwise anisotropy r = √(r_x²+r_y²) is:
#
#   r(α, θ) = C(α)·sin²θ / [ A(α) + B(α)·sin²θ ]
#
# The maximum r is reached when sin²θ = 1  (molecule lies in the focal plane):
#
#   r_sat(α) = C(α) / [ A(α) + B(α) ]
#
# We measure r_sat from data, invert the equation to get α, then NA = n·sin(α).
# r_sat is monotonically DECREASING with α (higher α → lower r_sat).
#
# Strategy: take the global max of r_max / r_99.5% / r_peak across all files,
# fit one α per metric, then verify by back-computing r_sat from the fitted NA.

from scipy.optimize import brentq

nw     = 1.33   # refractive index of immersion medium
NA_obj = 1.3    # nominal objective NA (reference only)

def fourkas_ABC(alpha):
    """Fourkas (2001) coefficients A, B, C as a function of collection half-angle α."""
    ca = np.cos(alpha)
    A  = 1/6  - ca/4        + (ca**3)/12
    B  =        ca/8        - (ca**3)/8
    C  = 7/48 - ca/16  - (ca**2)/16 - (ca**3)/48
    return A, B, C

def r_of_alpha_sinsq(alpha, sinsq_theta):
    """General r for a dipole at sin²θ collected with half-angle α.
       r = C·s / [A + B·s]  where s = sin²θ."""
    A, B, C = fourkas_ABC(alpha)
    s = sinsq_theta
    return C * s / (A + B * s)

def r_sat_theory(alpha):
    """r at sin²θ = 1 (molecule in focal plane) — the maximum achievable r.
       r_sat = C / (A + B).  Decreasing function of α."""
    return r_of_alpha_sinsq(alpha, 1.0)

# ── Reference values ─────────────────────────────────────────────────────────
alpha_ref  = np.arcsin(NA_obj / nw)
A_r, B_r, C_r = fourkas_ABC(alpha_ref)
r_sat_ref  = r_sat_theory(alpha_ref)

print(f"Reference: NA={NA_obj}, n={nw}  →  α={np.degrees(alpha_ref):.3f}°  r_sat={r_sat_ref:.5f}")
print(f"  A={A_r:.5f}  B={B_r:.5f}  C={C_r:.5f}")
print(f"  r_sat = C / (A+B) = {C_r:.5f} / {A_r+B_r:.5f} = {r_sat_ref:.5f}\n")

# ── Per-file r stats table ────────────────────────────────────────────────────
print(f"  {'File':<44} {'r_max':>8} {'r_99.5%':>9} {'r_peak':>8}")
print("  " + "-" * 72)
for tdms_path, st in r_stats_all.items():
    print(f"  {tdms_path.name:<44} {st['r_max']:>8.5f} {st['r_995']:>9.5f} {st['r_peak']:>8.5f}")

# ── Global maxima across all files ───────────────────────────────────────────
global_r_max  = max(st['r_max']  for st in r_stats_all.values())
global_r_995  = max(st['r_995']  for st in r_stats_all.values())
global_r_peak = max(st['r_peak'] for st in r_stats_all.values())

# r_sat is DECREASING with α: large α → small r_sat
r_hi = r_sat_theory(1e-3)           # small α  → high r_sat
r_lo = r_sat_theory(np.pi/2 - 1e-4) # α → 90° → low  r_sat

print(f"\n  r_sat valid range: [{r_lo:.5f}, {r_hi:.5f}]")

# ── Calibration helper ────────────────────────────────────────────────────────
def calibrate_from_r(r_sat_measured, label):
    r_clamped = float(np.clip(r_sat_measured, r_lo + 1e-8, r_hi - 1e-8))
    if abs(r_clamped - r_sat_measured) > 1e-9:
        print(f"  WARNING: {label} r_sat={r_sat_measured:.5f} clamped to [{r_lo:.5f}, {r_hi:.5f}]")
    alpha_fit = brentq(lambda a: r_sat_theory(a) - r_clamped,
                       1e-3, np.pi/2 - 1e-4, xtol=1e-9, rtol=1e-9)
    NA_fit = nw * np.sin(alpha_fit)
    A_f, B_f, C_f = fourkas_ABC(alpha_fit)
    r_verify = r_sat_theory(alpha_fit)   # back-calculate — should equal r_sat_measured
    status = "OK" if abs(r_verify - r_sat_measured) < 1e-6 else f"MISMATCH Δ={r_verify-r_sat_measured:.2e}"
    print(f"\n  {label}:")
    print(f"    r_sat_measured = {r_sat_measured:.5f}")
    print(f"    α_fit          = {np.degrees(alpha_fit):.4f}°")
    print(f"    NA_eff         = {NA_fit:.5f}  (n={nw})")
    print(f"    A={A_f:.5f}  B={B_f:.5f}  C={C_f:.5f}")
    print(f"    Verification   → r_sat_theory(α_fit) = {r_verify:.5f}  [{status}]")
    return dict(r_sat_measured=r_sat_measured, alpha_fit_deg=np.degrees(alpha_fit),
                NA_fit=NA_fit, A=A_f, B=B_f, C=C_f, r_verify=r_verify)

# ── Three global calibrations ─────────────────────────────────────────────────
print(f"\n{'='*60}")
print("Global calibration (max across all files)")
print(f"{'='*60}")
cal_r_max  = calibrate_from_r(global_r_max,  "r_max  (global max)")
cal_r_995  = calibrate_from_r(global_r_995,  "r_99.5% (global max)")
cal_r_peak = calibrate_from_r(global_r_peak, "r_peak  (global max)")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  {'Metric':<12} {'r_sat':>8} {'α (°)':>9} {'NA_eff':>8} {'r_verify':>10}")
print(f"  {'-'*12} {'-'*8} {'-'*9} {'-'*8} {'-'*10}")
for lbl, cal in [("r_max",   cal_r_max),
                 ("r_99.5%", cal_r_995),
                 ("r_peak",  cal_r_peak)]:
    print(f"  {lbl:<12} {cal['r_sat_measured']:>8.5f} {cal['alpha_fit_deg']:>9.4f}"
          f" {cal['NA_fit']:>8.5f} {cal['r_verify']:>10.5f}")
print(f"{'='*70}")
print(f"\n  Reference: NA={NA_obj}, n={nw}  →  α={np.degrees(alpha_ref):.4f}°  r_sat={r_sat_ref:.5f}")
